# Linear Programming in Linopy: Two-Plant Dispatch

## Objectives

At the end of this notebook you should be able to:

- write the two-plant dispatch problem from last week as a **linear program (LP)** in Linopy
- solve it with the HiGHS solver and read the optimal generation and total cost
- read the electricity **price** from the solver as the **shadow price** of the demand constraint
- show that the LP gives the same answer as the merit order

Last week you solved this problem with pen and paper: two power plants, one hour, one demand. You drew the feasible region and found the cheapest corner. Today you hand exactly the same problem to a solver. The numbers will not change — what changes is that the solver can handle problems far too big to draw.

In [1]:
import linopy
import pandas as pd

### The problem in energy terms

Here are our two plants. **Marginal cost** is what it costs to produce one more MWh (€/MWh), and **capacity** is the most a plant can produce in one hour (MW). Demand in our hour is 1,000 MW.

In [2]:
plants = pd.DataFrame(
    {"marginal_cost": [30, 80], "p_max": [800, 500]},
    index=pd.Index(["lignite", "gas"], name="plant"),
)
demand_MW = 1000
plants

         marginal_cost  p_max
plant                        
lignite             30    800
gas                 80    500

Every part of the LP has an energy meaning:

| LP term | Energy meaning |
|---|---|
| **decision variables** | how much each plant generates (MW) |
| **objective** | total generation cost (€), which we minimise |
| **constraints** | physical limits: capacity, and supply = demand |

Writing this table down before coding is a good habit: if you can't fill it in, you aren't ready to code the model yet.

### Step 1: create the model and the variables

A Linopy `Model` is an empty container. We add one **variable** per plant: generation must be between 0 MW and the plant's capacity. Passing the plant names as coordinates gives every variable a readable label.

In [3]:
m = linopy.Model()

generation = m.add_variables(
    lower=0,
    upper=plants["p_max"],
    coords=[plants.index],
    name="generation",
)
generation

Variable (plant: 2)
-------------------
[lignite]: generation[lignite] ∈ [0, 800]
[gas]: generation[gas] ∈ [0, 500]

Notice that Linopy read the upper bounds straight from the `p_max` column: lignite can produce at most 800 MW and gas at most 500 MW. These bounds are our capacity constraints.

### Step 2: add the demand constraint

Supply must equal demand: the generation of both plants adds up to exactly 1,000 MW. We give the constraint a **name** — we will need it later to read the price.

In [4]:
m.add_constraints(generation.sum() == demand_MW, name="demand")

Constraint `demand`
-------------------
+1 generation[lignite] + 1 generation[gas] = 1000.0

### Step 3: add the objective

The objective is total cost: generation of each plant (MW, over one hour = MWh) times its marginal cost (€/MWh). Linopy minimises by default.

In [5]:
m.add_objective((generation * plants["marginal_cost"]).sum())
m

Linopy LP model

Variables:
----------
 * generation (plant)

Expressions:
------------
<empty>

Constraints:
------------
 * demand

Status:
-------
initialized

### Step 4: solve

In [6]:
m.solve(solver_name="highs")

Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
LP linopy-problem-72fcd1ef has 1 row; 2 cols; 2 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [3e+01, 8e+01]
  Bound   [5e+02, 8e+02]
  RHS     [1e+03, 1e+03]
Presolving model
0 rows, 0 cols, 0 nonzeros 0s
0 rows, 0 cols, 0 nonzeros 0s
Presolve reductions: rows 0(-1); columns 0(-2); nonzeros 0(-2) - Reduced to empty
Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-72fcd1ef
Model status        : Optimal
Objective value     :  4.0000000000e+04
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00


('ok', 'optimal')

The solver reports `ok` and `optimal`: it found the cheapest possible dispatch. Let's look at the result in a small table.

In [7]:
result = generation.solution.to_pandas().rename("generation_MW").to_frame()
result

         generation_MW
plant                 
lignite          800.0
gas              200.0

In [8]:
total_cost = m.objective.value
print(f"Total cost: {total_cost:,.0f} €")

Total cost: 40,000 €


Lignite runs at its full 800 MW because it is the cheapest plant, and gas covers the remaining 200 MW. The total cost is 800 × 30 + 200 × 80 = 40,000 €. This is exactly the corner you found on paper last week.

### The merit order solves the same problem

In Module 02 you dispatched plants with the **merit order**: sort by marginal cost and fill demand from the cheapest plant up. Here is a short version of that function so we can compare.

In [9]:
def merit_order_dispatch(plants, demand_MW):
    """Return generation (MW) per plant and the clearing price (€/MWh)."""
    dispatch = {}
    remaining = demand_MW
    price = None
    for name, row in plants.sort_values("marginal_cost").iterrows():
        produced = min(row["p_max"], max(remaining, 0))
        dispatch[name] = produced
        if produced > 0:
            price = row["marginal_cost"]
        remaining -= produced
    return pd.Series(dispatch), price

In [10]:
mo_dispatch, mo_price = merit_order_dispatch(plants, demand_MW)
result["merit_order_MW"] = mo_dispatch
result

         generation_MW  merit_order_MW
plant                                 
lignite          800.0             800
gas              200.0             200

The two columns are identical. The merit order was secretly solving this LP all along: for a single hour with only capacity limits, filling up from the cheapest plant *is* the optimal solution. The LP becomes more useful next week, when we add constraints the merit order cannot handle.

### The price is a shadow price

Every constraint has a **shadow price** (also called its **dual value**): how much the total cost would rise if the constraint were tightened by one unit. For the demand constraint that means: *what does one more MW of demand cost?* That is exactly the electricity price.

In [11]:
price = float(m.constraints["demand"].dual)
print(f"Shadow price of demand: {price:.0f} €/MWh")
print(f"Merit-order price:      {mo_price:.0f} €/MWh")

Shadow price of demand: 80 €/MWh
Merit-order price:      80 €/MWh


One more MW of demand would be met by gas, which costs 80 €/MWh — so the price is 80 €/MWh. The solver derived the market price mathematically, without us telling it which plant is "marginal".

### Try other demand levels

Let's wrap the model in a function so we can solve it for different demand levels and compare it with the merit order each time.

In [12]:
def lp_dispatch(plants, demand_MW):
    """Solve the one-hour dispatch LP. Return (generation per plant, total cost, price)."""
    m = linopy.Model()
    g = m.add_variables(lower=0, upper=plants["p_max"], coords=[plants.index], name="generation")
    m.add_constraints(g.sum() == demand_MW, name="demand")
    m.add_objective((g * plants["marginal_cost"]).sum())
    m.solve(solver_name="highs", output_flag=False)
    return g.solution.to_pandas(), m.objective.value, float(m.constraints["demand"].dual)

In [13]:
rows = []
for d in [500, 1000, 1200]:
    gen, cost, lp_price = lp_dispatch(plants, d)
    _, mo_p = merit_order_dispatch(plants, d)
    rows.append({"demand_MW": d, "lignite_MW": gen["lignite"], "gas_MW": gen["gas"],
                 "total_cost_EUR": cost, "LP_price_EUR_per_MWh": lp_price,
                 "merit_order_price_EUR_per_MWh": mo_p})
pd.DataFrame(rows)

   demand_MW  lignite_MW  gas_MW  total_cost_EUR  LP_price_EUR_per_MWh  \
0        500       500.0     0.0         15000.0                  30.0   
1       1000       800.0   200.0         40000.0                  80.0   
2       1200       800.0   400.0         56000.0                  80.0   

   merit_order_price_EUR_per_MWh  
0                             30  
1                             80  
2                             80  

At 500 MW only lignite runs and the price is 30 €/MWh. Once demand is above 800 MW, gas is needed and the price jumps to 80 €/MWh. In every row the LP and the merit order agree.

## Check your understanding

**Reading the model**

1. Before running anything: what will the dispatch, total cost and price be for a demand of 800 MW exactly? Write your answer down.
2. Run `lp_dispatch(plants, 800)`. Does the price match your guess? (Hint: at exactly 800 MW, lignite is full. Is the next MW lignite or gas?)

**Changing the data**

1. Add a third plant, `"nuclear"`, with a marginal cost of 10 €/MWh and a capacity of 300 MW. Re-run the comparison table. What changes, and why?
2. What happens if demand is 1,400 MW (more than both plants together)? Run it and read the solver status. What would a real market do?

**Shadow prices**

1. In your own words: why is the shadow price of the demand constraint the electricity price?
2. If gas became cheaper than lignite (say 25 €/MWh), which plant would set the price at 1,000 MW of demand? Check your answer with the model.